# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities (record sets, fields, columns) by their `@id`.

### Dataset Source
The dataset source is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata and check description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
This section reviews the available record sets and their fields and columns using their `@id` fields.

We list record sets, their fields, and selected columns from the package to identify available entities and data for subsequent steps.

In [ ]:
print("Available record sets (@id):\n")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"- {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        print("    Available fields (@id):")
        for f in fields:
            if '@id' in f:
                print(f"      - {f['@id']} (name: {f.get('name', f.get('@id'))})")
            elif isinstance(f, str):
                print(f"      - {f}")
    if 'column' in rs:
        columns = rs['column']
        print("    Available columns (@id):")
        for c in columns:
            if '@id' in c:
                print(f"      - {c['@id']} (name: {c.get('name', c.get('@id'))})")
            elif isinstance(c, str):
                print(f"      - {c}")

if not record_set_ids:
    print("No record sets found in metadata -- attempting to discover via dataset.records().\n")
    # Try collecting record set ids from records method iterators
    # This is a fallback in case direct record_sets metadata is missing.
    # Since the record sets are missing from metadata, let's attempt to load and print available record_set ids
    # However, mlcroissant may require at least one correct @id, so the next cell attempts individual load.

### Print available records for a given record set (@id)

To inspect record data, you need to use a valid `record_set` @id. If available from above, use it directly; otherwise, examine the records to try reasonable guesses (e.g., the dataset may expose raw and regression record sets).

In [ ]:
# List possible record set @ids for access
# Direct access can sometimes be tricky if the schema uses complex nested structures. In many Croissant datasets
# the main record sets are accessible via their @id or via the names in metadata.record_sets. If none found, skip ahead.
from itertools import islice

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Preview records from record set @id: {example_record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(json.dumps(rec, indent=2))
        if i >= 2:
            break
else:
    # Fallback: Try a well-known default record set id pattern for MLCommons Croissant datasets
    # For some packages, one can try 'mlcroissant:records' or similar.
    print("No record sets detected in metadata. Please refer to the documentation or schema for available record set @ids.")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Always use the record set and field `@id`s as discovered above.

Below, we use the discovered record set `@id`s, or you can manually supply known ones.

In [ ]:
# If record_set_ids is empty, you may need to manually insert the correct @id from the schema documentation.
# Replace with discovered or known record set @ids as necessary.
import warnings

if not record_set_ids:
    # Fallback: attempt with a likely record set @id -- replace as needed
    # e.g., 'cr:OrderedLogisticRegressionResults' or similar. Please adapt to your dataset as necessary.
    record_set_ids = ['cr:OrderedLogisticRegressionResults']
    warnings.warn("No record set @ids auto-discovered. Replace 'cr:OrderedLogisticRegressionResults' with the correct value if needed.")

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded records: {len(dataframes[rs_id])}\n")

# Display the columns for the first record set
main_record_set_id = record_set_ids[0]
print(f"Available fields/columns in DataFrame for record set @id '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering records based on a numeric field (by its `@id`), normalizing that field, and grouping by a categorical field (also referenced by its `@id`).

Please adjust the `numeric_field_id` and `group_field_id` variables to fields discovered from the record set overview above, using their exact `@id` values.

In [ ]:
# Adjust these as per available field @ids in your dataset.
# Example field @ids -- replace with those printed above.
numeric_field_id = 'cr:log_likelihood'     # e.g., for log-likelihood column
group_field_id = 'cr:ward_name'            # e.g., for group by ward or region
record_set_id = main_record_set_id

df = dataframes[record_set_id]

if numeric_field_id in df.columns:
    threshold = 10  # Change threshold as appropriate for your dataset
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by categorical field if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print(f"Column {group_field_id} not found in the DataFrame for grouping.")
else:
    print(f"Numeric field {numeric_field_id} not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare groups defined by the group field (by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group field
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² dataset using the `mlcroissant` library.

- All entities (record sets, fields, columns) were referenced using their `@id`.
- We loaded record set data into Pandas DataFrames and performed numeric filtering, normalization, and group-wise summarization.
- Data distributions and relationships were visualized.

This workflow can be adapted to other Croissant-structured datasets by referencing the appropriate `@id` fields for your analysis needs.